In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import scanpy as sc
import torch
import pandas as pd
import numpy as np
import random
from scipy.sparse import issparse
import SpaDiff as sd

In [ ]:
parser = sd.get_args() 
args, unknown = parser.parse_known_args() 
args.n_clusters = 5
args.domains = 5
print(args.random_seed)
torch.manual_seed(args.random_seed)
torch.cuda.manual_seed_all(args.random_seed)
np.random.seed(args.random_seed)
random.seed(args.random_seed)
torch.backends.cudnn.deterministic = True

if args.cuda >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{args.cuda}")
    torch.cuda.manual_seed_all(args.random_seed)
    print(f"Using GPU: {torch.cuda.get_device_name(args.cuda)}")
else:
    device = torch.device("cpu")
    print("Using CPU.")

In [ ]:
id = "151672"
adata = sc.read_visium("../0_data/case1/"+id+"/") 
adata.var_names_make_unique()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000, subset=True)
truth = pd.read_csv('../0_data/case1/'+id+'/truth.txt', sep='\t', header=None, index_col=0)
truth.columns = ['Truth']
adata.obs['Truth'] = truth.loc[adata.obs_names, 'Truth']

In [ ]:
adata

In [ ]:
adata, HL = sd.spatial_reconstruction(adata, n_neighbors=args.n_neighbors, n_pcs =15, alpha=args.rec_alpha)

In [ ]:
sc.pp.pca(adata, n_comps=50)
features = torch.tensor(adata.obsm['X_pca'].toarray() if issparse(adata.obsm['X_pca']) else np.array(adata.obsm['X_pca'])).float()

In [ ]:
clf=sd.DEC( features,HL,
              node_width=features.shape[1],
              device = device,
              opt="adam", 
              args=args)

In [ ]:
a = clf.fit(features, HL)
y_pred, prob, z =clf.predict()
adata.obsm['z'] = z

In [ ]:
adata.obs["mclust"] = sd.utils.mclust_R(adata, used_obsm='z',pca_num = 25 , num_cluster=args.domains)
adata.obs['mclust'] = adata.obs['mclust'].astype('int')
adata.obs['mclust'] = adata.obs['mclust'].astype('category')

In [ ]:
adata

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score
obs_df = adata.obs.dropna()
ARI = adjusted_rand_score(obs_df["mclust"], obs_df['Truth'])
print('ARI of mclust = %.3f' %ARI)

In [ ]:
plot_color = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
sc.pl.spatial(adata,
              img_key="hires",
              color="mclust",
              palette=plot_color,  
              title="ARI=%.2f"%ARI,
              legend_loc=None,
              frameon=False,
              spot_size=120,
              show=False
              )